In [1]:
from datasets import load_dataset
import re
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from transformers import T5Tokenizer , T5ForConditionalGeneration
from nltk.corpus import stopwords
from nltk.tokenize import sent_tokenize
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

d:\GitUploads\Case-Classification\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lalit\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lalit\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
ds = load_dataset("ninadn/indian-legal")

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['Text', 'Summary'],
        num_rows: 7030
    })
    test: Dataset({
        features: ['Text', 'Summary'],
        num_rows: 100
    })
})

In [4]:
train_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

In [5]:
train_df

,Text,Summary
0,Appeal No. LXVI of 1949.\nAppeal from the High...,The charge created in respect of municipal pro...
1,Civil Appeal No.94 of 1949.\n107 834 Appeal fr...,"An agreement for a lease, which a lease is by ..."
2,"iminal Appeal No. 40 of 1951, 127 Appeal from ...","The question whether a Magistrate is ""personal..."
3,Appeal No. 388 of 1960.\nAppeal by special lea...,The appellant was a member of a joint Hindu fa...
4,Appeal No. 198 of 1954.\nAppeal from the judgm...,The appellant was the Ruler of the State of Ba...
...,...,...
7025,Appeal No. 761 of 1957.\nAppeal by special lea...,The respondent company purchased certain machi...
7026,Appeal No. 761 of 1957.\nAppeal by special lea...,The respondent company purchased certain machi...
7027,Appeal No. 53 of 1958.\nAppeal by special leav...,"While this appeal by special leave, relating t..."
7028,Appeal No. 4 of 1960.\nAppeal by special leave...,The Punjab Government issued notification unde...


In [6]:
print(train_df['Summary'].value_counts().head(10))

Summary
The respondents filed a suit for specific performance against the appellant which was dismissed on March 12, 1954.\nOn March 24 the respondents made an application for a certified copy of the judgment and decree.\nThe decree was not drawn up and the respondents were supplied a certified copy of the judgment and the memo of costs.\nThe respondents filed an appeal before the High Court without the certified copy of the decree and only with the certified copy of the judgment and the memo of costs.\nThe appeal was admitted under 0.\n41, r. 11 Code of Civil Procedure on August 30, 1954.\nOn December 23, 1958, the appellant served a notice on the respondents that he would raise a preliminary objection at the hearing that the appeal was incompetent as a certified copy of the decree was not filed as required by 0. 41, r. 1.\nOn December 24, 1958, the respondents moved the trial Court for drawing up of the decree, but since the record was in the High Court this could not be done.\nAt th

In [7]:
stop_words = set(stopwords.words('english'))

In [8]:
def clean(text):
    if text is None:
        return ''
    text = re.sub(r'\d+','',text)
    text = re.sub(r'\n',' ',text)
    text = re.sub(r'\s+',' ',text)
    text = re.sub(r'[^\w\s]','',text)
    text = text.lower()
    text = ' '.join(word for word in text.split() if word not in stop_words)
    return text.strip()

In [9]:
train_df['Text'] = train_df['Text'].apply(clean)
train_df['Summary'] = train_df['Summary'].apply(clean)
test_df['Text'] = test_df['Text'].apply(clean)
test_df['Summary'] = test_df['Summary'].apply(clean)

In [10]:
train_df

,Text,Summary
0,appeal lxvi appeal high court judicature bomba...,charge created respect municipal property tax ...
1,civil appeal appeal judgment decree high court...,agreement lease lease indian declared include ...
2,iminal appeal appeal judgment order dated st j...,question whether magistrate personally interes...
3,appeal appeal special leave judgment order dat...,appellant member joint hindu family carried bu...
4,appeal appeal judgment order dated october for...,appellant ruler state baster later integrated ...
...,...,...
7025,appeal appeal special leave judgment order dat...,respondent company purchased certain machinery...
7026,appeal appeal special leave judgment order dat...,respondent company purchased certain machinery...
7027,appeal appeal special leave decision dated feb...,appeal special leave relating industrial dispu...
7028,appeal appeal special leave judgment order dat...,punjab government issued notification sections...


In [11]:
model_name = 't5-small'
Tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [12]:
print(train_df.columns)


Index(['Text', 'Summary'], dtype='object')


In [13]:
max_input_length = 256
max_target_length = 128

def encode(example):
    input_text = 'summarize: ' + example['Text']
    target_text = example['Summary']
    inputs_ids = Tokenizer.encode(
        input_text, max_length = max_input_length, truncation =True
    )
    target_ids = Tokenizer.encode(
        target_text , max_length = max_target_length,truncation = True
    )    
    return {'input_ids':inputs_ids,'labels':target_ids}

train_df = train_df.apply(encode,axis =1)
test_df = test_df.apply(encode,axis =1)

In [14]:
data_collator = DataCollatorForSeq2Seq(tokenizer=Tokenizer, model=model)


training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    weight_decay=0.01,
    save_total_limit=1,
    logging_dir="./logs",
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_df,
    eval_dataset=test_df,
    tokenizer=Tokenizer,
    data_collator=data_collator
)


d:\GitUploads\Case-Classification\.venv\lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\lalit\AppData\Local\Temp\ipykernel_9588\3980112742.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
trainer.train()


 14%|█▍        | 500/3515 [17:06<1:25:36,  1.70s/it]

{'loss': 5.0622, 'grad_norm': 14.052681922912598, 'learning_rate': 1.7155049786628736e-05, 'epoch': 0.14}


 28%|██▊       | 1000/3515 [31:27<1:08:07,  1.63s/it]

{'loss': 4.6932, 'grad_norm': 3.1709091663360596, 'learning_rate': 1.431009957325747e-05, 'epoch': 0.28}


 43%|████▎     | 1500/3515 [45:08<55:18,  1.65s/it]  

...

{'loss': 4.5594, 'grad_norm': 2.979816198348999, 'learning_rate': 8.5348506401138e-08, 'epoch': 1.0}


100%|██████████| 3515/3515 [1:45:07<00:00,  1.82s/it]

{'train_runtime': 6307.9844, 'train_samples_per_second': 1.114, 'train_steps_per_second': 0.557, 'train_loss': 4.65688675382568, 'epoch': 1.0}


100%|██████████| 3515/3515 [1:45:08<00:00,  1.79s/it]


TrainOutput(global_step=3515, training_loss=4.65688675382568, metrics={'train_runtime': 6307.9844, 'train_samples_per_second': 1.114, 'train_steps_per_second': 0.557, 'total_flos': 475726432174080.0, 'train_loss': 4.65688675382568, 'epoch': 1.0})

In [16]:
model.save_pretrained("./legal_summarizer_model")
Tokenizer.save_pretrained("./legal_summarizer_model")

('./legal_summarizer_model\\tokenizer_config.json',
 './legal_summarizer_model\\special_tokens_map.json',
 './legal_summarizer_model\\spiece.model',
 './legal_summarizer_model\\added_tokens.json')

In [20]:
ds = load_dataset("ninadn/indian-legal")
test_df = ds['test'].to_pandas()

In [21]:
raw_test_df = test_df
def summarize(text):
    input_ids = Tokenizer.encode("summarize: " + text, return_tensors="pt", max_length=512, truncation=True)
    output = model.generate(input_ids, max_length=128, num_beams=4, early_stopping=True)
    return Tokenizer.decode(output[0], skip_special_tokens=True)

print(summarize(raw_test_df.iloc[0]["Text"]))


appellants who were admittedly displaced persons were granted quasi permanent allotment of 40 standard acres and 15 3/4 units of land in the village of Raikot in 1949. their father Sardar Nand Singh made an application for consolidation of his lands with those of the appellants in the village Raikot.
